# Chronos-2 (zero-shot)

Modèle pré-entraîné d'Amazon : pas d'entraînement, il lit les 2048 dernières heures avant `t0` de chaque arc et prolonge la série. On lui donne en plus le calendrier connu à l'avance (`COV`). `X_train` / `y_train` ne servent pas : on garde la même structure que les autres notebooks.

In [ ]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, FEATURES

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [ ]:
from chronos import Chronos2Pipeline
from dataset_create import ajouter_futur, vraies_valeurs

pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
fe, _ = ajouter_futur(df, "debit", vraies_valeurs(df).datetime.max())  # même index que les X de split / test_final -> date de chaque ligne
COV = ["ferie", "pont", "vacances", "paris_respire", "evenement", "confinement", "type_jour"]
CTX = 2048  # on prédit de t0 à la fin du test, à partir des 2048 h avant t0

def modele(X_train, y_train, X_test, cible="debit"):
    t0 = fe.loc[X_train.index, "datetime"].max() + pd.Timedelta("1h")
    H = int((fe.loc[X_test.index, "datetime"].max() - t0) / pd.Timedelta("1h")) + 1  # de t0 à la dernière heure testée
    g = fe.assign(item_id=fe.axe.astype(str), vacances=fe.vacances.cat.codes, timestamp=fe.datetime.dt.tz_convert(None))
    g = g[["item_id", "timestamp", cible, *COV]].astype({c: float for c in COV})
    ctx = g[(fe.datetime < t0) & (fe.datetime >= t0 - pd.Timedelta(hours=CTX))]
    fut = g[(fe.datetime >= t0) & (fe.datetime < t0 + pd.Timedelta(hours=H))].drop(columns=cible)
    c = pipe.predict_df(ctx, future_df=fut, target=cible, prediction_length=H, quantile_levels=[0.5])
    test = pd.DataFrame({"item_id": X_test.axe.astype(str), "timestamp": fe.loc[X_test.index, "datetime"].dt.tz_convert(None)})
    return test.merge(c[["item_id", "timestamp", "predictions"]], how="left").predictions.values

In [ ]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test, CIBLE)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

In [ ]:
# Les 3 tests communs, contre le naïf lag_7j : semaine de Lucie (07/11/2025), test final, backtest sur TOUS les vendredis
from evaluation import trois_tests
P = trois_tests(df, lambda X_train, y_train, X_test: modele(X_train, y_train, X_test, CIBLE), "chronos", CIBLE)